# 02 — Preprocessing (20% Sample Version)

This notebook performs full preprocessing on a **20% stratified sample** of the training dataset.  
It prepares:

- `processed_train_20.csv`  
- `processed_test_20.csv`  
- `train_ids_20.csv`  
- `test_ids_20.csv`  

This version is used for training faster models for analysis and experimentation.


### Import Libraries

In [3]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
pd.set_option("display.max_columns", None)

### Load Raw Train & Test Data

In [19]:
train = pd.read_csv("../data/raw/train_updated.csv")
test = pd.read_csv("../data/raw/test_updated.csv")
# Stratified 20% sampling based on RiskFlag
groups = []

for label, group in train.groupby("RiskFlag"):
    sampled = group.sample(frac=0.20, random_state=42)
    groups.append(sampled)

train_20 = pd.concat(groups).reset_index(drop=True)
print("Original train shape:", train.shape)
print("Sampled 20% Train Shape:", train_20.shape)
train_20["RiskFlag"].value_counts(normalize=True)

Original train shape: (204277, 18)
Sampled 20% Train Shape: (40856, 18)


RiskFlag
0    0.883714
1    0.116286
Name: proportion, dtype: float64

### Drop ID Column

In [20]:
# Save IDs before ANY transformations
train_ids = train_20["ProfileID"].copy()
test_ids  = test["ProfileID"].copy()

# Drop the ID column from feature set
train_20 = train_20.drop("ProfileID", axis=1)
test  = test.drop("ProfileID", axis=1)

### Identify columns as Numerical or Categorical or Binary

In [21]:
# Numeric features
numerical = [
    "ApplicantYears", "AnnualEarnings", "RequestedSum", "TrustMetric",
    "WorkDuration", "ActiveAccounts", "OfferRate", "RepayPeriod", "DebtFactor"
]

# Categorical features
categorical = [
    "QualificationLevel",
    "WorkCategory",
    "RelationshipStatus",
    "FundUseCase"
]

# Binary Yes/No features
binary_cols = ["OwnsProperty", "FamilyObligation", "JointApplicant"]

### Convert BInary Yes/No to 0/1

In [22]:
binary_map = {"Yes": 1, "No": 0}

for col in binary_cols:
    train_20[col] = train_20[col].map(binary_map)
    test[col] = test[col].map(binary_map)

### One-Hot Encode Categorical Columns (Train + Test Together)

In [23]:
# One-hot encode train
train_20_encoded = pd.get_dummies(train_20, columns=categorical, drop_first=True)

# One-hot encode test
test_encoded = pd.get_dummies(test, columns=categorical, drop_first=True)

# Save target
y = train_20_encoded["RiskFlag"].copy()

# Drop target temporarily
X_train20 = train_20_encoded.drop("RiskFlag", axis=1)
X_test = test_encoded.copy()

# Align columns
X_test_aligned = X_test.reindex(columns=X_train20.columns, fill_value=0)
X_train20_aligned = X_train20.copy()  # already correct

processed_train = X_train20_aligned.copy()
processed_train["RiskFlag"] = y.values

processed_test = X_test_aligned.copy()

### Save Processed Files

In [25]:
os.makedirs("../data/processed/processed_20", exist_ok=True)

processed_train.to_csv("../data/processed/processed_20/processed_train_20.csv", index=False)
processed_test.to_csv("../data/processed/processed_20/processed_test_20.csv", index=False)

train_ids.to_csv("../data/processed/processed_20/train_ids_20.csv", index=False)
test_ids.to_csv("../data/processed/processed_20/test_ids_20.csv", index=False)

print("Saved all processed 20% files:")
print("- processed_train_20.csv")
print("- processed_test_20.csv")
print("- train_ids_20.csv")
print("- test_ids_20.csv")

Saved all processed 20% files:
- processed_train_20.csv
- processed_test_20.csv
- train_ids_20.csv
- test_ids_20.csv
